# Three-species gastric antrum: data preparation and h5ad conversion

This notebook prepares human, mouse, and pig gastric-antrum count matrices, matches cells to `data/three_species_gastric_antrum_barcode_cell_type.csv`, performs basic QC, and writes:

- `processed_h5ad/human.h5ad`
- `processed_h5ad/mouse.h5ad`
- `processed_h5ad/pig.h5ad`

Only barcodes present in the annotation CSV are retained. Cell types are stored in `adata.obs['cell_type']`.

## Data source

The study is NCBI GEO **GSE225275**, "Cross-species single-cell transcriptomic analysis reveals the landscape of animal gastric antrum".

- GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE225275
- BioProject/SRA entry linked from GEO: PRJNA934889

GEO provides the study and raw-sequencing accessions, but does not currently list the following prepared expression matrices as supplementary files:

- `GSE225275_human_data.txt.gz`
- `GSE225275_mouse_data.txt.gz`
- `GSE225275_pig_data.txt.gz`

Therefore, use one of the two preparation options below. Option A copies the already prepared matrices from another local directory. Option B downloads them from direct URLs supplied by your lab or data host.

In [ ]:
from pathlib import Path
import subprocess
import sys

VIGNETTE_DIR = Path.cwd().resolve()
if VIGNETTE_DIR.name != '3_species_gatric':
    VIGNETTE_DIR = Path(r'D:\111icde_addition_experiments\MetaGeneFormer\Vignettes\3_species_gatric')

DATA_DIR = VIGNETTE_DIR / 'data'
OUTPUT_DIR = VIGNETTE_DIR / 'processed_h5ad'
ANNOTATION_CSV = DATA_DIR / 'three_species_gastric_antrum_barcode_cell_type.csv'

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Vignette:', VIGNETTE_DIR)
print('Annotation exists:', ANNOTATION_CSV.exists())

## Option A: copy prepared matrices from an existing local directory

This does not move or delete the source data. It copies the three compressed matrices into this vignette's `data` directory. Skip this cell if the files are already present.

In [ ]:
SOURCE_DIR = Path(r'D:\111icde_addition_experiments\3_species_gastric')

subprocess.run([
    sys.executable,
    str(VIGNETTE_DIR / 'download_three_species_gastric_data.py'),
    '--source-dir', str(SOURCE_DIR),
    '--data-dir', str(DATA_DIR),
], check=True)

## Option B: download prepared matrices from direct URLs

Uncomment and fill all three URLs if your lab, cloud storage, Figshare, Zenodo, or another repository hosts the processed matrices. Do not run both Option A and Option B unless `--overwrite` is intended.

In [ ]:
# HUMAN_URL = 'https://example.org/GSE225275_human_data.txt.gz'
# MOUSE_URL = 'https://example.org/GSE225275_mouse_data.txt.gz'
# PIG_URL = 'https://example.org/GSE225275_pig_data.txt.gz'
#
# subprocess.run([
#     sys.executable,
#     str(VIGNETTE_DIR / 'download_three_species_gastric_data.py'),
#     '--data-dir', str(DATA_DIR),
#     '--human-url', HUMAN_URL,
#     '--mouse-url', MOUSE_URL,
#     '--pig-url', PIG_URL,
# ], check=True)

## Verify input files

The expression tables must be gzip-compressed, tab-separated, gene-by-cell matrices. The first row contains barcodes and the first column contains gene names.

In [ ]:
required_files = [
    DATA_DIR / 'GSE225275_human_data.txt.gz',
    DATA_DIR / 'GSE225275_mouse_data.txt.gz',
    DATA_DIR / 'GSE225275_pig_data.txt.gz',
    ANNOTATION_CSV,
]

for path in required_files:
    print(path.name, 'OK' if path.exists() else 'MISSING', path.stat().st_size if path.exists() else '')

missing = [path for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

## Convert to annotated, QC-filtered h5ad

Default filters:

- keep only cells present in the annotation CSV;
- at least 200 detected genes per cell;
- mitochondrial count percentage at most 20%;
- retain genes detected in at least 3 retained cells.

The raw count matrix remains sparse and unnormalized in `adata.X`. QC values are stored in `obs`; cell type is stored in `obs['cell_type']`.

In [ ]:
subprocess.run([
    sys.executable,
    str(VIGNETTE_DIR / 'prepare_three_species_gastric_h5ad.py'),
    '--input-dir', str(DATA_DIR),
    '--annotation-csv', str(ANNOTATION_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--min-genes', '200',
    '--min-cells', '3',
    '--max-pct-mt', '20',
], check=True)

## Validate output

In [ ]:
import anndata as ad

for species in ('human', 'mouse', 'pig'):
    path = OUTPUT_DIR / f'{species}.h5ad'
    adata = ad.read_h5ad(path, backed='r')
    assert 'cell_type' in adata.obs.columns
    assert adata.obs_names.is_unique
    print(species, adata.shape, adata.obs['cell_type'].value_counts().to_dict())
    adata.file.close()